In [ ]:
#python -m venv openai-env
#openai-env\Scripts\activate
#API设置：https://platform.openai.com/docs/quickstart
#pip install ipykernel
#python -m ipykernel install --user --name=openai-env --display-name "Python (openai-env)"
#jupyter notebook
#需要绑定付款方式：https://platform.openai.com/settings/organization/billing/overview

In [1]:
#!pip install --upgrade openai

In [2]:
!pip show openai

Name: openai
Version: 0.27.4
Summary: Python client library for the OpenAI API
Home-page: https://github.com/openai/openai-python
Author: OpenAI
Author-email: support@openai.com
License: 
Location: c:\users\yurul\anaconda3\lib\site-packages
Requires: aiohttp, requests, tqdm
Required-by: 


In [ ]:
import base64
import requests
from openai import OpenAI
import os
import re

In [ ]:
api_key=os.environ.get("OPENAI_API_KEY")
print(api_key)

In [ ]:
# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# analyze images in folder

In [ ]:
import os
import csv
import requests
import base64
import re
import time

# Function to encode image to base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Function to analyze an image and return the category and confidence
def analyze_image(image_path, api_key, timeout=10):  # Add timeout to requests.post later, not as a function argument
    base64_image = encode_image(image_path)
    
    # Define the 19 categories for classification
    categories = [
        "Text poster: Poster with main text and simple visual elements related to the topic, often using symbols or graphics to emphasize the message. main visual elements: text, symbols, graphics.",
        "Poster with persons: Poster with text and a half-length frontal shot of one or several persons in a professional or semi-formal. It must include text",
        "Group photos: Group photos featuring people in professional or semi-formal attire, taken both indoors and outdoors, often in a social or networking context.",
        "Interviews: Photos of interviews, panel discussions, or broadcasts involving cameras and microphones, often in studio or on-set environments.",
        "Speech: Photos of political speeches, discussions, and interviews, typically featuring individuals at podiums, panels, or in debate settings.",
        "Campaign: Political campaign events and public speeches involving groups or individuals with banners, posters, and symbols representing their parties or causes.",
        "Meetings: Individuals in professional or semi-formal attire, often engaged in public speaking, discussions, or gatherings, set against backdrops such as podiums or banners.",
        "Military and International Relations: Focus on political and international themes, such as military involvement or diplomatic activities.",
        "Newspaper: News reports related to politics, featuring figures in formal or official contexts, with visual elements such as newspapers or tablets.",
        "Outdoor Scenes and Activities: Images include outdoor settings, often with natural backdrops like forests, sports field, lake, boat, or fields, and people in casual activities like biking.",
        "Food: Focus on food, people engaged in social activities, often accompanied by smiles and festive settings.",
        "Infographics: Bold, attention-grabbing titles, with charts, graphs, maps, and icons to communicate statistical information or trends.",
        "Public Service in Action: Individuals wearing safety or work uniforms, interacting in public spaces or workplaces.",
        "Social media posts: Screenshots of tweet-like formatted text posts, with bright backgrounds, typically containing political statements or opinions.",
        "Natural landscapes: Depictions of natural landscapes or scenery, featuring elements like water bodies, clouds, trees, and greenery.",
        "Animals: Images focusing on animals, often in natural or casual outdoor settings, sometimes with people interacting with them.",
        "Celebratory or special occasions: Warm and celebratory atmosphere with decorations like flowers, candles, and festive interactions.Or it is related to the sacrificial scene",
        "selfies: Individuals as the main focus, often in casual or natural poses, with backgrounds providing context.",
        "Solo shots: include a mix of formal attire, direct and indirect eye contact with the camera, as well as both frontal and side-profile shots. The settings often feature neutral or minimal backgrounds, and the lighting—either natural or studio—emphasizes a professional or serious atmosphere.It doesn't include any text",
        "Community engagement: Photos include outdoor settings (e.g. on the street), people interacting with others in a positive or engaging way (such as smiles, giving flowers, or casual conversations). Additionally, some photos feature interactions with vulnerable groups, such as the elderly, children, or people with disabilities. These elements suggest a friendly, social atmosphere with a focus on community engagement.",
        "Other topics: Doesn't belong to any of the categories above."
    ]

    # Prepare the prompt
    prompt_text = "Classify the following image into one of these categories:\n\n"
    for idx, category in enumerate(categories, 1):
        prompt_text += f"{idx}. {category}\n"

    prompt_text += "\nPlease return only the number and title (the text before ':') of the best matching category, followed by a confidence score between 0 and 1, separated by a comma. For example: '3. Group photos, 0.85'."

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    # Payload with the image and the new classification prompt
    payload = {
        "model": "gpt-4o",  # Use the specified model name
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt_text
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        "max_tokens": 300
    }

    # Send the request to the OpenAI API with a timeout
    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload, timeout=timeout)
    
    # Check for successful response
    if response.status_code != 200:
        print(f"Error: Received status code {response.status_code}")
        print(response.text)  # Print the full response for debugging
        return None, None

    response_json = response.json()

    # Check if 'choices' exists in the response
    if 'choices' not in response_json:
        print(f"Error: 'choices' not found in the response: {response_json}")
        return None, None

    # Extract the content from the response
    content = response_json['choices'][0]['message']['content']

    # Use regex to extract category and confidence
    match = re.match(r"\d+\.\s*([^,]+),\s*([0-9]*\.?[0-9]+)", content)
    if match:
        category = match.group(1).strip()
        confidence = float(match.group(2))
        return category, confidence
    else:
        print(f"Unexpected response format: {content}")
        return None, None

# Function to read existing image names from the CSV file
def get_existing_images(output_csv):
    existing_images = set()  # Use a set for faster lookup
    if os.path.exists(output_csv):
        with open(output_csv, mode='r', encoding='utf-8') as file:
            reader = csv.reader(file)
            next(reader)  # Skip the header row
            for row in reader:
                if row:
                    existing_images.add(row[0])  # Add the image filename to the set
    return existing_images

# Function to process the image and retry in case of network failure
def analyze_image_with_retry(image_path, api_key, retries=3, delay=5, timeout=10):
    for attempt in range(retries):
        try:
            # 调用 analyze_image 函数处理图片，并设置超时时间
            category, confidence = analyze_image(image_path, api_key, timeout=timeout)
            if category is not None and confidence is not None:
                return category, confidence
            else:
                print(f"Failed to get valid response for {os.path.basename(image_path)}.")
                return None, None
        except requests.exceptions.Timeout:
            # 如果是网络超时，等待一段时间并重试
            print(f"Request timed out for {os.path.basename(image_path)}. Retrying {attempt + 1}/{retries} after {delay} seconds...")
            time.sleep(delay)
        except requests.exceptions.RequestException as e:
            # 捕获其他网络错误
            print(f"Network error: {e}. Retrying {attempt + 1}/{retries} after {delay} seconds...")
            time.sleep(delay)
        except Exception as e:
            # 捕获其他错误
            print(f"Error processing {os.path.basename(image_path)}: {e}")
            return None, None
    return None, None  # 如果所有尝试都失败，返回None


# Function to process images in a folder and save results to CSV
def process_images(folder_path, api_key, output_csv):
    # Check if the file already exists to determine whether to write the header
    file_exists = os.path.exists(output_csv)

    # Get the set of existing images in the CSV file
    existing_images = get_existing_images(output_csv)

    # Open the CSV file in append mode
    with open(output_csv, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # If the file doesn't exist, write the header
        if not file_exists:
            writer.writerow(["Image", "Category", "Confidence"])

        # Loop through the files in the folder
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                # Check if the image has already been processed
                if filename not in existing_images:
                    image_path = os.path.join(folder_path, filename)
                    try:
                        # Process the image and get the category and confidence (with retry)
                        category, confidence = analyze_image_with_retry(image_path, api_key)
                        if category and confidence:
                            writer.writerow([filename, category, confidence])
                            print(f"Processed {filename}: {category}, Confidence: {confidence}")
                        else:
                            print(f"Failed to process {filename} after retries.")
                    except Exception as e:
                        print(f"Error processing {filename}: {e}")
                # 如果你仍然需要处理已存在的图片但不打印跳过信息，可以在这里添加代码
                # 否则，已跳过的图片将不会被处理

# Example usage:
# process_images("F:\\phd data\\post images", "your_openai_api_key", "F:\\test\\test.csv")


In [5]:
# Main execution
if __name__ == "__main__":
    # Retrieve the API key from the environment variable
    api_key = os.getenv('OPENAI_API_KEY')

    # Ensure the API key is set
    if not api_key:
        raise ValueError("OpenAI API key is not set in the environment variable 'OPENAI_API_KEY'")

    # Define the folder path and output CSV file
    #folder_path = r"F:\phd data\post images"
    folder_path = r'F:\phd data\other topic'
    output_csv = r"C:\coding\jupyternotebook\phd project\results\visual topic results-other topics.csv"

    # Process the images and save the results to CSV
    process_images(folder_path, api_key, output_csv)

Processed abaerbock_Ca73MmKNuVM.jpg: Other topics, Confidence: 0.8
Processed abaerbock_CNzH5WMnSK_.jpg: Celebratory or special occasions, Confidence: 0.75
Processed abaerbock_CQ0ymvPKyda.jpg: Solo shots, Confidence: 0.85
Processed abaerbock_CQA7mUPqCEW.jpg: Solo shots, Confidence: 0.75
Processed abaerbock_CQB_1sTCNDe.jpg: Solo shots, Confidence: 0.85
Processed abaerbock_CQJcr5pqlzy.jpg: Other topics, Confidence: 0.95
Processed abaerbock_CQVaImMCNRG.jpg: Solo shots, Confidence: 0.9
Processed abaerbock_CSM5I05qBgO.jpg: Other topics, Confidence: 0.75
Processed abaerbock_CUSXloZssR4.jpg: Public Service in Action, Confidence: 0.8
Processed abaerbock_CUudzuTKGI8.jpg: Solo shots, Confidence: 0.95
Processed abaerbock_CVSzTchK62Q.jpg: Other topics, Confidence: 0.7
Processed adis.a93_CN0mLQNMu7M.jpg: Other topics, Confidence: 0.85
Processed adis.a93_COqMV09tOHJ.jpg: Solo shots, Confidence: 0.95
Processed adis.a93_CP3gqWvtoVm.jpg: Other topics, Confidence: 0.75
Processed adis.a93_CPa17_ONsZj.jpg:

Processed cem.oezdemir_CWLEuLqKfVi.jpg: Other topics, Confidence: 0.75
Processed christianlindner_CaWa0WsMdky.jpg: Solo shots, Confidence: 0.9
Processed christianlindner_CNDjvPAh0lO.jpg: Solo shots, Confidence: 0.9
Processed christianlindner_CONjZLqhDAO.jpg: Other topics, Confidence: 0.7
Processed christianlindner_COR_WW-hTfs.jpg: Solo shots, Confidence: 0.9
Processed christianlindner_CQejizeh6H3.jpg: Other topics, Confidence: 0.9
Processed christianlindner_CQvNfoVpatG.jpg: Meetings, Confidence: 0.75
Processed christianlindner_CR9TN9Xop0e.jpg: Community engagement, Confidence: 0.75
Processed christianlindner_CRMWSw8IqIK.jpg: Solo shots, Confidence: 0.9
Processed christianlindner_CRtyas8Iau-.jpg: Other topics, Confidence: 0.95
Processed christianlindner_CRWahoRIrzS.jpg: Solo shots, Confidence: 0.75
Processed christianlindner_CSBpA12IOJw.jpg: Solo shots, Confidence: 0.85
Processed christianlindner_CSJQmz4Iz6b.jpg: Solo shots, Confidence: 0.75
Processed christianlindner_CVpUDZOoSpM.jpg: S

Processed janine_wissler_CNVFd-lsJvL.jpg: Text poster, Confidence: 0.85
Processed janine_wissler_CPdIzIBNHkW.jpg: Natural landscapes, Confidence: 0.75
Processed janine_wissler_CPnyByWN964.jpg: Text poster, Confidence: 0.8
Processed janine_wissler_CPvtcJvt2Lo.jpg: Other topics, Confidence: 0.9
Processed janine_wissler_CRJE30osu3n.jpg: Solo shots, Confidence: 0.9
Processed janine_wissler_CV-C2opM5gQ.jpg: Other topics, Confidence: 0.95
Processed jankorte77_CaIWJ-st2Yg.jpg: Other topics, Confidence: 0.9
Processed jankorte77_CajsLzQsNwJ.jpg: Other topics, Confidence: 0.75
Processed jankorte77_CaujV3rtcLp.jpg: Newspaper, Confidence: 0.7
Processed jankorte77_CaxftY9Nhv-.jpg: Natural landscapes, Confidence: 0.75
Processed jankorte77_CbfERESNhNY.jpg: Other topics, Confidence: 0.9
Processed jankorte77_CbiVCOdtIk_.jpg: Other topics, Confidence: 0.85
Processed jankorte77_CbS6CIVNLbM.jpg: Other topics, Confidence: 0.85
Processed jankorte77_CN2OrKKMdIP.jpg: Other topics, Confidence: 0.95
Processed j

Processed juliakloeckner_CQEbdnfhIYm.jpg: Other topics, Confidence: 0.75
Processed juliakloeckner_CQoz7r0BtXP.jpg: Community engagement, Confidence: 0.75
Processed juliakloeckner_CRNCsscpWYr.jpg: Text poster, Confidence: 0.9
Processed juliakloeckner_CRYWtvLpitC.jpg: Other topics, Confidence: 0.75
Processed juliakloeckner_CS4KVUeonX9.jpg: Speech, Confidence: 0.75
Processed juliakloeckner_CS8_jHyoIQq.jpg: Other topics, Confidence: 0.95
Processed juliakloeckner_CT5Z92zI7T9.jpg: Other topics, Confidence: 0.8
Processed juliakloeckner_CTaMQASIubV.jpg: Other topics, Confidence: 0.75
Processed juliakloeckner_CTEcf9cIzAu.jpg: Community engagement, Confidence: 0.75
Processed juliakloeckner_CTMSi3vIahp.jpg: Community engagement, Confidence: 0.75
Processed juliakloeckner_CTnYjwEISNz.jpg: Community engagement, Confidence: 0.85
Processed juliakloeckner_CTR1R68INLc.jpg: Other topics, Confidence: 0.85
Processed juliakloeckner_CTShOiSo3zq.jpg: Solo shots, Confidence: 0.9
Processed juliakloeckner_CTt92L

Processed larsklingbeil_CZHZJi2Nk_-.jpg: Solo shots, Confidence: 0.9
Processed lindateuteberg_CSgu-yho5NO.jpg: Solo shots, Confidence: 0.85
Processed lindateuteberg_CVf8SoEIRC3.jpg: Other topics, Confidence: 0.95
Processed maas.heiko_COgGAxctnPw.jpg: Solo shots, Confidence: 0.8
Processed maas.heiko_COpNLkgNQGK.jpg: Solo shots, Confidence: 0.95
Processed maas.heiko_CRi9eGKMMj6.jpg: Solo shots, Confidence: 0.95
Processed maas.heiko_CTErVVbMC1k.jpg: Other topics, Confidence: 0.75
Processed maas.heiko_CV-cMO6s1Ls.jpg: Other topics, Confidence: 0.95
Processed maas.heiko_CVfWvqzsHkH.jpg: Other topics, Confidence: 0.9
Processed maas.heiko_CVPVtS2s1_5.jpg: Solo shots, Confidence: 0.75
Processed maas.heiko_CW6STfKtuds.jpg: Other topics, Confidence: 0.9
Processed maas.heiko_CWH_Qf9tas0.jpg: Solo shots, Confidence: 0.9
Processed maas.heiko_CYZc0wot2V-.jpg: Solo shots, Confidence: 0.95
Processed malte.kaufmann_Cab8mXYLz6h.jpg: Other topics, Confidence: 0.75
Processed malte.kaufmann_CQEOKU5NgWF.jpg

Processed mike_mohring_CODeCIjhPsI.jpg: Solo shots, Confidence: 0.9
Processed mike_mohring_CQeKk51hp7Z.jpg: Natural landscapes, Confidence: 0.7
Processed mike_mohring_CQvZPY7JPwg.jpg: Natural landscapes, Confidence: 0.7
Processed mike_mohring_CR7FhORoJyT.jpg: Other topics, Confidence: 0.95
Processed mike_mohring_CRmASCComgf.jpg: Other topics, Confidence: 0.75
Processed mike_mohring_CT-dY_9o7UE.jpg: Celebratory or special occasions, Confidence: 0.8
Processed mike_mohring_CTClf-7odCT.jpg: Poster with persons, Confidence: 0.7
Processed mike_mohring_CTIHAPLIeO5.jpg: Other topics, Confidence: 0.95
Processed mike_mohring_CTjTZCJoyiU.jpg: Other topics, Confidence: 0.9
Processed mike_mohring_CTNZlqUojUZ.jpg: Other topics, Confidence: 0.75
Processed mike_mohring_CU193KzIGzd.jpg: Celebratory or special occasions, Confidence: 0.75
Processed mike_mohring_CUDj7SKoEJU.jpg: Food, Confidence: 0.95
Processed mike_mohring_CUM9PkAohf3.jpg: Other topics, Confidence: 0.95
Processed mike_mohring_CXxDaqEIjP4

Processed petr.bystron_CViUu2MtCWB.jpg: Other topics, Confidence: 0.9
Processed petr.bystron_CW5THPFNkIY.jpg: Other topics, Confidence: 0.9
Processed petr.bystron_CXEXTzTMe8m.jpg: Solo shots, Confidence: 0.75
Processed petr.bystron_CXNtEjhNoSd.jpg: Other topics, Confidence: 0.75
Processed petr.bystron_CYzeIZiNEML.jpg: Natural landscapes, Confidence: 0.7
Processed philipp.amthor_CNKt3tjhfX0.jpg: Other topics, Confidence: 0.75
Processed philipp.amthor_CPTdxffBw8y.jpg: Outdoor Scenes and Activities, Confidence: 0.75
Processed philipp.amthor_CQ9P09go_24.jpg: Outdoor Scenes and Activities, Confidence: 0.95
Processed philipp.amthor_CRgk40yJEPv.jpg: Outdoor Scenes and Activities, Confidence: 0.85
Processed philipp.amthor_CSKG_AYIVl1.jpg: Outdoor Scenes and Activities, Confidence: 0.9
Processed philipp.amthor_CUkk6cKIEfp.jpg: Other topics, Confidence: 0.75
Processed philipp.amthor_CZ6v9hYoQs7.jpg: Other topics, Confidence: 0.75
Processed philipp.amthor_CZujAcwI1Au.jpg: Solo shots, Confidence: 

Processed toni_hofreiter_CTbvhVqsabx.jpg: Natural landscapes, Confidence: 0.8
Processed toni_hofreiter_CTe4cOvsmDc.jpg: Solo shots, Confidence: 0.9
Processed toni_hofreiter_CTjQcR2Mfip.jpg: Natural landscapes, Confidence: 0.75
Processed toni_hofreiter_CTxIWi-MGvK.jpg: Other topics, Confidence: 0.9
Processed toni_hofreiter_CVAbr1qtjNZ.jpg: Solo shots, Confidence: 0.9
Processed toni_hofreiter_CVzujT_M_yQ.jpg: Solo shots, Confidence: 0.9
Processed volkerwissing_CYbC1xzIU-F.jpg: Solo shots, Confidence: 0.75
Processed _svenlehmann__CaWlko3sq_5.jpg: Other topics, Confidence: 0.9
Processed _svenlehmann__CODs2KFMYPv.jpg: Other topics, Confidence: 0.95
Processed _svenlehmann__COp_qJHNj00.jpg: Other topics, Confidence: 0.9
Processed _svenlehmann__CQa20X7Nj8Z.jpg: Other topics, Confidence: 0.95
Processed _svenlehmann__CQG88hetHPZ.jpg: Other topics, Confidence: 0.9
Processed _svenlehmann__CRJPTM-MTJu.jpg: Solo shots, Confidence: 0.9
Processed _svenlehmann__CRV9OsbMz_p.jpg: Outdoor Scenes and Activ